## Substitutes

In [ ]:
import pandas as pd
%run ../utils/substitutes.py

file_path_root = "../data/validation/substitutes/"

# Load files
product_df = pd.read_csv('../data/cleaned/product-info-full.csv')
orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')

Run a qualitative sanity check for chosen items: 
Coke Classic  (ID: 16696)
Banana  (ID: 24852)
Eggo Homestyle Waffles  (ID: 30696)

In [3]:
focus_products = [16696,24852,30696]

dept1_df = pd.read_csv('../data/cleaned/pairwise-dept1.csv')
dept7_df = pd.read_csv('../data/cleaned/pairwise-dept7.csv')
dept4_df = pd.read_csv('../data/cleaned/pairwise-dept4.csv')

pairwise_df = pd.concat([dept1_df, dept4_df, dept7_df], ignore_index=True)

similarity_df = pd.read_csv('../data/cleaned/product-similiarity.csv')

compute_sub_score(
    product_df,
    pairwise_df,
    focus_products,
    similarity_df,
    f"{file_path_root}minimal-scored-subs.csv")

Completed substitute calculations. Saved to ../data/validation/substitutes/minimal-scored-subs.csv


In [5]:
%run ../utils/results.py

substitutes_df = pd.read_csv(f"{file_path_root}minimal-scored-subs.csv")

best_threshold = 0.4
# Add a new column 'identified_substitute' based on the threshold
substitutes_df['identified_substitute'] = substitutes_df['score'] >= best_threshold

# Compute transferability pct
results = compute_transferability(substitutes_df, top_n=5)

# Print the results
show_sub_results(results)


Product: Coke Classic  (ID: 16696)


,sub_name,transferability_pct,score,aisle
0,Classic Soda,0.149826,0.701174,soft drinks
1,Coke Zero,0.144124,0.674490,soft drinks
2,Coke,0.141851,0.663852,soft drinks
3,Cherry Coke,0.134633,0.630075,soft drinks
4,Vanilla Coke Zero,0.130740,0.611854,soft drinks



Product: Banana  (ID: 24852)


,sub_name,transferability_pct,score,aisle
0,Bananas,0.177190,0.724160,fresh fruits
1,Organic Banana,0.170498,0.696808,fresh fruits
2,Baby Bananas,0.162533,0.664258,fresh fruits
3,Bag of Organic Bananas,0.107661,0.440000,fresh fruits
4,Organic Strawberries,0.106278,0.434348,fresh fruits



Product: Eggo Homestyle Waffles  (ID: 30696)


,sub_name,transferability_pct,score,aisle
0,Homestyle Waffles,0.145661,0.699007,frozen breakfast
1,Homestyle Belgian Waffles,0.141890,0.680906,frozen breakfast
2,Eggo Buttermilk Waffles,0.138502,0.664652,frozen breakfast
3,Eggo Thick & Fluffy Original Waffles,0.138212,0.663260,frozen breakfast
4,Buttermilk Waffles,0.134741,0.646601,frozen breakfast


Calculate a hybrid substitution score for the sampled products and save to CSV.

In [18]:
sampled_products_df = pd.read_csv('../data/validation/sampled-products.csv')
products_list = sampled_products_df['product_id'].tolist()

for i in range(3, 22):
    pairwise_df = pd.read_csv(f'../data/validation/pairwise/pairwise-dept{i}.csv')
    filtered_df = product_df[
        (product_df['department_id'] == i) &
        (product_df['product_id'].isin(products_list))
    ]
    sampled_products = filtered_df['product_id'].tolist()

    compute_sub_score(
        product_df,
        pairwise_df,
        sampled_products,
        similarity_df,
        f"{file_path_root}dept{i}-scored-subs.csv")

Completed substitute calculations. Saved to ../data/validation/substitutes/dept3-scored-subs.csv
Completed substitute calculations. Saved to ../data/validation/substitutes/dept4-scored-subs.csv
Completed substitute calculations. Saved to ../data/validation/substitutes/dept5-scored-subs.csv
Completed substitute calculations. Saved to ../data/validation/substitutes/dept6-scored-subs.csv
Completed substitute calculations. Saved to ../data/validation/substitutes/dept7-scored-subs.csv
Completed substitute calculations. Saved to ../data/validation/substitutes/dept8-scored-subs.csv
Completed substitute calculations. Saved to ../data/validation/substitutes/dept9-scored-subs.csv
Completed substitute calculations. Saved to ../data/validation/substitutes/dept10-scored-subs.csv
Completed substitute calculations. Saved to ../data/validation/substitutes/dept11-scored-subs.csv
Completed substitute calculations. Saved to ../data/validation/substitutes/dept12-scored-subs.csv
Completed substitute calcul

Filter the substitutes from the previous step based on calculated threshold and mark as identified substitutes. Save to CSV.

Calculate a transferability % for the identified substitutes. Save to CSV.

In [19]:
best_threshold = 0.4

for i in range(1, 22):
    substitutes_df = pd.read_csv(f"{file_path_root}dept{i}-scored-subs.csv")

    # Add a new column 'identified_substitute' based on the threshold
    substitutes_df['identified_substitute'] = substitutes_df['score'] >= best_threshold

    results = compute_transferability(substitutes_df, top_n=5)
    results.to_csv(f"{file_path_root}dept{i}-transfer.csv", index=False)

Validate the results from the previous steps.

In [11]:
%run ../utils/substitutes.py

transfer_df = pd.read_csv(f'{file_path_root}transfer.csv')

orders_df = orders_full_df[['order_id', 'product_id']]

results = run_multiple_subsets_validation(
    orders_df=orders_df,
    subs_df=substitutes_df,
    transfer_df=transfer_df, 
    n_subsets=20,
    sample_size=200,
    method="freq_stratified",   # "random" or "freq_stratified"
    p_switch=0.5,
    K=5,
    n_trials=50,
    seed=2025
)

print(results)
print("Aggregated means:")
print(results.mean(numeric_only=True))

KeyboardInterrupt: 